# Fundas_System -- AMD Specialist Training (Phase 3, Huong B, plans/plan_AMD.txt)

FUNDAS_SYSTEM -- TRAIN AMD (THOAI HOA DIEM VANG) TREN GOOGLE COLAB
====================================================
Boi canh: plans/plan_AMD.txt muc 6.3 (Huong B: tu train, khong dung
DeepSeeNet pretrained -- nguoi dung chon huong nay vi nhanh hon va dong bo
kien truc PyTorch/timm voi toan bo he thong, khong can convert tu TF1/Keras
cu). Model nay la phan loai NHI PHAN (amd / normal), CUNG mau voi
plans/Fundas_Glaucoma_Training.ipynb (1 backbone timm + 1 logit + sigmoid,
KHONG ensemble).

Dataset: AMDNet23 (Kaggle: orvile/macular-degeneration-disease-dataset,
nguon goc Mendeley: Ali 2025, doi 10.17632/yj35kjgrv3.1). License CC BY 4.0.
Chi dung 2 trong 4 thu muc cua bo nay: amd/ va normal/ (bo qua diabetes/ va
cataract/, khong lien quan bai toan nay).

DA XAC NHAN BANG ANH THAT (2026-09-01, xem plans/plan_AMD.txt muc 2) truoc
khi viet notebook nay -- rut kinh nghiem tu vu ACRIMA/RIM-ONE (glaucoma) bi
crop sai va vu dataset cataract 4-lop bi sai modality:
  - CA 2 thu muc amd/ va normal/ DEU la anh FUNDUS TRON DAY DU (co vien
    den, thay dia thi giac + mach mau), DUNG dinh dang can, KHONG bi
    crop-zoom nhu ACRIMA. An toan de dung thang inference/preprocessing.py
    hien co, KHONG can them buoc auto-crop nhu glaucoma.

3 RUI RO DA BIET, XU LY NGAY TRONG NOTEBOOK NAY (tranh lap lai loi cu):
  (1) TRUNG LAP FILE: kiem tra thu muc goc thay ro rang trung lap kieu
      "102_left.jpg" VA "102_left (1).jpg" (cung anh, 2 ten khac nhau).
      Neu khong loc, 1 anh co the vua nam trong train vua nam trong val
      -> ro ri du lieu NGAY TRONG CHINH lan split nay (khac han rui ro
      ODIR o muc (2) duoi day). Cell 4 loc trung lap bang HASH NOI DUNG
      FILE (MD5), khong chi dua vao ten file.
  (2) NGUON GOC MOT PHAN TU ODIR-5K: ~60% ten file trong ca amd/ va
      normal/ theo dung quy uoc "<so>_left.jpg"/"<so>_right.jpg" cua chinh
      ODIR-5K (dataset da dung train model multidisease). Day KHONG chan
      viec train 1 model MOI, RIENG BIET nhu the nay (model nay khac model
      multidisease) -- nhung CAN BIET để khong nham day la nguon du lieu
      "doc lap 100%". Cell 4 GAN NHAN THEM 1 co "likely_odir_source" (suy
      tu ten file) chi de BAO CAO rieng, KHONG dung de loai bo anh.
  (3) ANH DA QUA TIEN XU LY MAU (Gamma correction, CLAHE -- theo mo ta
      chinh thuc cua AMDNet23) TRUOC KHI chung ta nhan duoc, trong khi
      inference/preprocessing.py (dang dung cho DR/multidisease/glaucoma)
      KHONG lam buoc nay voi anh nguoi dung upload that. Co the gay lech
      phan phoi mau sac giua luc train (anh da CLAHE) va luc suy luan that
      (anh tho). KHONG co cach khac phuc trong notebook nay (khong biet
      tham so CLAHE/gamma chinh xac ho dung) -- GHI RO trong config.json
      va PLAN KIEM CHUNG END-TO-END SAU (giong da lam voi glaucoma +
      AIROGS-light) TRUOC KHI tin tuong con so accuracy o day cho anh
      nguoi dung that.

Output cua notebook nay (2 file tai ve rieng, tu dong khi chay het cell):
  1. model_best.pth -- checkpoint tot nhat (chon theo val F1 lop amd).
  2. amd_test_samples.zip -- 10 anh ngau nhien (5 amd + 5 normal) tu tap
     validation, kem labels.json, de kiem thu cuc bo sau khi viet xong
     inference/amd_model.py + inference/amd_engine.py.

CHUAN BI TRUOC KHI CHAY:
  1. Tai kaggle.json len Google Drive (vd /content/drive/MyDrive/kaggle.json).
  2. Colab: Runtime > Change runtime type > GPU.
  3. Chay lan luot tung cell TU TREN XUONG.

## CELL 1: Mount Drive + cau hinh Kaggle API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials ready.")

## CELL 2: Cai dependency + tai dataset AMDNet23

In [ ]:
!pip install -q kaggle timm

import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    'orvile/macular-degeneration-disease-dataset',
    path='/content/amd_raw',
    unzip=True,
    quiet=False,
)
print("Dataset downloaded.")

## CELL 3: Imports + tien xu ly (khop inference/preprocessing.py)

Ham crop/normalize duoi day la ban sao chinh xac cua inference/
preprocessing.py -- KHONG duoc tu tien doi logic o day ma khong doi ca file
local, neu khong checkpoint se lech phan phoi du lieu luc suy luan that.

In [ ]:
import hashlib
import json
import random
import re
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Duong dan thuc te sau khi giai nen (thu muc long nhau, ten co dau cach
# kep "for  Age-Related" -- giu nguyen dung nhu Kaggle dong goi).
RAW_ROOT = Path(
    '/content/amd_raw/'
    'AMDNet23 Fundus Image Dataset for Age-Related Macular Degeneration Disease Detection/'
    'AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/'
    'AMDNet23 Dataset/train'
)

BACKBONE = 'seresnext50_32x4d'   # dung lai backbone da co trong DR/multidisease/glaucoma
IMAGE_SIZE = 224
DROPOUT = 0.3
VAL_FRACTION = 0.15
SEED = 0
EPOCHS = 25                       # tap nho hon glaucoma (~900 anh) -- them epoch, theo doi overfit qua val_f1
BATCH_SIZE = 32
LR = 3e-4
N_TEST_SAMPLES = 10                # 5 amd + 5 normal, xuat o Cell 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def crop_dark_borders(img: np.ndarray, tol: int = 7) -> np.ndarray:
    """Y het inference/preprocessing.py::_crop_dark_borders."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if mask.sum() == 0:
        return img
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    cropped = img[rows[0]: rows[-1] + 1, cols[0]: cols[-1] + 1]
    return cropped if cropped.size else img


def load_and_crop(image_path) -> np.ndarray:
    """Y het inference/preprocessing.py::load_and_crop."""
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f'Could not read image: {image_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return crop_dark_borders(img)


_MEAN, _STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet fallback
try:
    cfg = timm.get_pretrained_cfg(BACKBONE)
    if cfg is not None and cfg.mean and cfg.std:
        _MEAN, _STD = cfg.mean, cfg.std
except Exception:
    pass

_TRANSFORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])


def preprocess(cropped_rgb: np.ndarray) -> torch.Tensor:
    """Y het inference/preprocessing.py::preprocess_for_backbone."""
    return _TRANSFORM(Image.fromarray(cropped_rgb))


# Chan doan nhanh (rut kinh nghiem glaucoma): in kich thuoc + xem thu vai
# anh moi lop TRUOC khi train, de phat hien som neu dinh dang khac gia
# dinh (da xac nhan bang mat truoc khi viet notebook nay -- xem plans/
# plan_AMD.txt muc 2 -- nhung van kiem tra lai o day phong khi ban tai ve
# tren Colab khac voi ban da kiem tra thu cong).
def _peek_sizes(folder, label, n=5):
    paths = sorted((RAW_ROOT / folder).glob('*'))[:n]
    sizes = [Image.open(p).size for p in paths if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f'  {label}: {sizes}')

print('Kich thuoc anh mau (W, H) moi lop (kiem tra nhanh dinh dang):')
_peek_sizes('amd', 'AMD')
_peek_sizes('normal', 'Normal')

## CELL 4: Xay manifest -- LOC TRUNG LAP (hash noi dung) + gan nhan proxy nguon

RUI RO DA BIET (xem docstring dau file): (1) file trung lap ten khac nhau
trong chinh dataset nay -- PHAI loc bang hash noi dung, khong chi bang ten,
neu khong 1 anh co the roi vao ca train VA val cung luc (ro ri ngay trong
lan split nay, nghiem trong hon ca chuyen ODIR o muc (2)).

In [ ]:
def file_hash(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()


ODIR_NAME_RE = re.compile(r'^\d+_(left|right)', re.IGNORECASE)


def load_class(folder: str, label: int):
    rows = []
    seen_hashes = set()
    n_total, n_dupe = 0, 0
    for p in sorted((RAW_ROOT / folder).iterdir()):
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        n_total += 1
        h = file_hash(p)
        if h in seen_hashes:
            n_dupe += 1
            continue
        seen_hashes.add(h)
        likely_odir = bool(ODIR_NAME_RE.match(p.name))
        rows.append((p, label, likely_odir))
    print(f'  {folder}: {n_total} files -> {len(rows)} unique after hash-dedup ({n_dupe} duplicates dropped)')
    return rows


print('Loading + deduplicating (by file content hash, not filename):')
amd_rows = load_class('amd', 1)
normal_rows = load_class('normal', 0)
all_rows = amd_rows + normal_rows

n_odir = sum(1 for _, _, is_odir in all_rows if is_odir)
print(f'\nTotal unique images: {len(all_rows)}  (amd={len(amd_rows)}, normal={len(normal_rows)})')
print(f'Likely-ODIR-5K-sourced (by filename pattern, informational only): {n_odir}/{len(all_rows)}')

assert len(all_rows) > 500, 'Expected roughly 800-1000 unique images total -- check RAW_ROOT path above if this fails.'

## CELL 5: Split train/val (stratify theo label) + Dataset/DataLoader

Khong co patient/eye ID doc lap qua duoc 6 nguon da tron trong AMDNet23
(giong tinh huong glaucoma voi ACRIMA/ORIGA/RIM-ONE) -- split o day la THEO
TUNG ANH (sau khi da loai trung lap o Cell 4), stratify theo label.

In [ ]:
def stratified_split(rows, val_fraction, seed):
    rng = random.Random(seed)
    groups = {}
    for row in rows:
        groups.setdefault(row[1], []).append(row)

    train_rows, val_rows = [], []
    for label, group in groups.items():
        group = group[:]
        rng.shuffle(group)
        n_val = max(1, round(len(group) * val_fraction))
        val_rows.extend(group[:n_val])
        train_rows.extend(group[n_val:])
    return train_rows, val_rows


train_rows, val_rows = stratified_split(all_rows, VAL_FRACTION, SEED)
print(f'Train: {len(train_rows)}  Val: {len(val_rows)}')
print('Val per label (1=amd):', {l: sum(1 for _, lb, _ in val_rows if lb == l) for l in (0, 1)})


class AmdDataset(Dataset):
    def __init__(self, rows, augment: bool):
        self.rows = rows
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        path, label, _is_odir = self.rows[idx]
        cropped = load_and_crop(path)
        if self.augment and np.random.rand() < 0.5:
            cropped = np.ascontiguousarray(cropped[:, ::-1])
        return preprocess(cropped), torch.tensor(label, dtype=torch.float32)


train_labels = np.array([r[1] for r in train_rows])
n_pos, n_neg = train_labels.sum(), len(train_labels) - train_labels.sum()
pos_weight = torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32)
print(f'Train pos(amd)={int(n_pos)} neg(normal)={int(n_neg)} pos_weight={pos_weight.item():.3f}')

train_loader = DataLoader(AmdDataset(train_rows, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(AmdDataset(val_rows, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## CELL 6: Model -- phan loai nhi phan (1 logit, sigmoid = P(amd))

Cung mau voi inference/glaucoma_model.py: 1 backbone timm + head Linear don,
KHONG ensemble (tap qua nho de train 20 model rieng nhu DR).

In [ ]:
class AmdModel(nn.Module):
    def __init__(self, backbone_name, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)  # raw logit


model = AmdModel(BACKBONE, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

## CELL 7: Vong lap train -- chon model tot nhat theo val F1 (lop amd)

Sau vong lap, in them breakdown theo "likely_odir_source" (proxy suy tu ten
file, xem Cell 4) -- KHONG phai de loai anh, chi de xem model co hoc lech
theo nguon khong (cung tinh than voi breakdown theo acrima/origa/rimone da
lam cho glaucoma).

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        probs = torch.sigmoid(model(x)).cpu()
        all_probs.append(probs)
        all_labels.append(y)
    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = (probs >= 0.5).astype(int)

    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (preds == labels).mean()
    return {'accuracy': float(accuracy), 'precision': precision, 'recall': recall, 'f1': f1}


def evaluate_by_source_proxy(model, rows):
    groups = {'likely_odir': [], 'other_source': []}
    for row in rows:
        groups['likely_odir' if row[2] else 'other_source'].append(row)
    results = {}
    for name, group_rows in groups.items():
        if not group_rows:
            continue
        loader = DataLoader(AmdDataset(group_rows, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        results[name] = (len(group_rows), evaluate(model, loader))
    return results


best_f1 = -1.0
best_state = None
best_metrics = {}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_rows)
    metrics = evaluate(model, val_loader)
    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f}  "
          f"val_acc={metrics['accuracy']:.3f}  val_precision={metrics['precision']:.3f}  "
          f"val_recall={metrics['recall']:.3f}  val_f1={metrics['f1']:.3f}")

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_metrics = {'val_accuracy': metrics['accuracy'], 'val_precision': metrics['precision'],
                         'val_recall': metrics['recall'], 'val_f1': metrics['f1'], 'epoch': epoch}

print(f"\nBest: val_f1={best_f1:.3f}  (epoch {best_metrics.get('epoch')})")

model.load_state_dict(best_state)
print('\nBreakdown by likely-source-proxy (best checkpoint, informational only):')
for name, (n, m) in evaluate_by_source_proxy(model, val_rows).items():
    print(f"  {name:14s} (n={n:3d})  acc={m['accuracy']:.3f}  precision={m['precision']:.3f}  recall={m['recall']:.3f}  f1={m['f1']:.3f}")

## CELL 8: Luu checkpoint + TU DONG tai ve may

Chay xong cell nay se tu dong: (1) luu vao Google Drive, (2) trigger tai ve
trinh duyet (`files.download`).

In [ ]:
CHECKPOINT_NAME = 'model_best.pth'
torch.save(
    {
        'model_state_dict': best_state,
        'labels': ['normal', 'amd'],   # index 1 = positive class cua sigmoid logit
        'backbone': BACKBONE,
        'dropout': DROPOUT,
        **best_metrics,
    },
    CHECKPOINT_NAME,
)

shutil.copy(CHECKPOINT_NAME, '/content/drive/MyDrive/' + CHECKPOINT_NAME)
print('Saved to Google Drive:', '/content/drive/MyDrive/' + CHECKPOINT_NAME)

from google.colab import files
files.download(CHECKPOINT_NAME)   # tu dong tai ve may

print(json.dumps({
    'task': 'AMD Specialist Screening (Phase 3, plans/plan_AMD.txt, Huong B)',
    'dataset_source': 'https://www.kaggle.com/datasets/orvile/macular-degeneration-disease-dataset (AMDNet23, Mendeley doi:10.17632/yj35kjgrv3.1)',
    'backbone': BACKBONE,
    'image_size': IMAGE_SIZE,
    'dropout': DROPOUT,
    'labels': ['normal', 'amd'],
    'checkpoint': CHECKPOINT_NAME,
    'is_experimental': True,
    'requires_optic_disc_crop': False,
    'known_caveats': [
        'Training images were pre-enhanced by the dataset author (gamma correction, CLAHE) before we received them -- inference/preprocessing.py does NOT replicate this, so a real user upload (unenhanced) may look different from what this model learned on. Needs end-to-end validation on genuinely raw fundus photos before trusting these numbers in production (same process as plans/plan_glaucoma.txt did with AIROGS-light).',
        'A majority of source images (see likely_odir_source breakdown printed above) appear to be drawn from ODIR-5K by filename convention -- this does not invalidate training a new dedicated model here, but means this dataset is not a fully independent population from the one behind the multidisease model.',
    ],
    **best_metrics,
}, indent=4))

## CELL 9: Xuat 10 anh ngau nhien, TY LE 5 amd / 5 normal (tu tap validation) de test local

Sample RIENG 5 tu nhom amd va 5 tu nhom normal (khong random tho tren ca
pool, de dam bao co du ca 2 lop) roi tron thu tu lai -- cung ky thuat da
dung o glaucoma. Copy ANH GOC (chua qua crop/resize).

In [ ]:
SAMPLE_DIR = Path('/content/amd_test_samples')
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)
SAMPLE_DIR.mkdir(parents=True)

rng = random.Random(SEED + 1)
amd_val_rows = [r for r in val_rows if r[1] == 1]
normal_val_rows = [r for r in val_rows if r[1] == 0]

n_each = N_TEST_SAMPLES // 2  # 5 + 5
sample_rows = (
    rng.sample(amd_val_rows, min(n_each, len(amd_val_rows)))
    + rng.sample(normal_val_rows, min(n_each, len(normal_val_rows)))
)
rng.shuffle(sample_rows)

label_names = {0: 'normal', 1: 'amd'}
manifest = []
for i, (path, label, is_odir) in enumerate(sample_rows, start=1):
    dest_name = f'{i:02d}_{label_names[label]}{path.suffix}'
    shutil.copy(path, SAMPLE_DIR / dest_name)
    manifest.append({'filename': dest_name, 'true_label': label_names[label], 'likely_odir_source': is_odir})

n_amd = sum(1 for m in manifest if m['true_label'] == 'amd')
n_normal = sum(1 for m in manifest if m['true_label'] == 'normal')

with open(SAMPLE_DIR / 'labels.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'{len(sample_rows)} sample images copied to {SAMPLE_DIR} (amd={n_amd}, normal={n_normal}):')
for m in manifest:
    print(' ', m)

ZIP_PATH = '/content/amd_test_samples.zip'
shutil.make_archive('/content/amd_test_samples', 'zip', SAMPLE_DIR)

files.download(ZIP_PATH)   # tu dong tai ve may
print(f'\nDownloaded: {ZIP_PATH}')

## SAU KHI TAI VE MAY LOCAL

1. Copy `model_best.pth` vua tai ve vao:
     D:\Fundas_System\models\amd\model_best.pth
   (tao thu muc `models/amd/` neu chua co)
2. Copy doan JSON in ra o Cell 9 (da co val_accuracy/val_precision/
   val_recall/val_f1 that + known_caveats) vao 1 file moi:
     D:\Fundas_System\config\amd\config.json
3. Giai nen `amd_test_samples.zip` vao 1 thu muc cuc bo (vd
   `test_images_amd/`) -- dung 10 anh + `labels.json` de kiem thu thu cong
   sau khi viet xong `inference/amd_model.py` + `inference/amd_engine.py`
   (theo dung mau da lam voi multidisease/glaucoma).
4. QUAN TRONG (xem canh bao Cell 9's known_caveats): TRUOC KHI tich hop vao
   web/, nen lam 1 vong kiem chung end-to-end tren anh fundus THAT CHUA QUA
   CLAHE/gamma (giong da lam voi glaucoma + AIROGS-light,
   scripts/validate_glaucoma_e2e.py) -- con so val_f1 o day do tren anh DA
   duoc tien xu ly mau boi tac gia dataset, CO THE khong dai dien dung cho
   anh nguoi dung upload that.